In [ ]:
# !pip install -q pandas numpy scikit-learn nltk matplotlib seaborn
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
DATA_URL = "https://www.datheon.in/datasets/sentiment_reviews.csv"
LOCAL_PATH = "sentiment_reviews.csv"   # download manually and place beside the notebook if the URL fails

try:
    df = pd.read_csv(DATA_URL)
    print("Loaded from URL")
except Exception as e:
    print("URL load failed:", e, "-> trying local file")
    df = pd.read_csv(LOCAL_PATH)

print(df.shape)
df.head(10)

In [ ]:
print(df.isna().sum())
df = df.dropna(subset=["review_text", "sentiment"]).drop_duplicates(subset="review_text")
df["sentiment"] = df["sentiment"].str.strip().str.lower()

print("\nClass distribution (all):")
print(df["sentiment"].value_counts())

# Binary task: keep only positive / negative
df = df[df["sentiment"].isin(["positive", "negative"])].reset_index(drop=True)
df["label"] = (df["sentiment"] == "positive").astype(int)   # 1 = positive, 0 = negative
print("\nBinary dataset:", df.shape)

df["sentiment"].value_counts().plot(kind="bar", color=["#4c9f70", "#d9534f"], title="Class balance")
plt.show()

In [ ]:
import nltk

def _try_download(pkg):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    _try_download(pkg)

# Stopwords: NLTK if available, otherwise scikit-learn's list
try:
    from nltk.corpus import stopwords
    BASE_STOPWORDS = set(stopwords.words("english"))
except Exception:
    BASE_STOPWORDS = set(ENGLISH_STOP_WORDS)
    print("NLTK stopwords unavailable -> using scikit-learn list")

NEGATIONS = {"no", "nor", "not", "never", "none", "nothing", "neither", "n't",
             "dont", "doesnt", "didnt", "isnt", "wasnt", "wont", "cant", "couldnt",
             "shouldnt", "wouldnt", "hasnt", "havent", "hadnt", "arent", "werent", "aint"}
STOPWORDS = BASE_STOPWORDS - NEGATIONS

# Lemmatizer: WordNet if available, otherwise no-op
try:
    from nltk.stem import WordNetLemmatizer
    _lem = WordNetLemmatizer()
    _lem.lemmatize("tests")      # triggers LookupError if wordnet is missing
    def lemmatize(tok):
        # try verb form first ("stopped" -> "stop"), then noun ("batteries" -> "battery")
        v = _lem.lemmatize(tok, pos="v")
        return v if v != tok else _lem.lemmatize(tok, pos="n")
except Exception:
    print("WordNet unavailable -> lemmatization skipped")
    lemmatize = lambda tok: tok

HTML_RE = re.compile(r"<[^>]+>")
CONTRACTION_RE = re.compile(r"n't\b")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")

def clean_text(text: str) -> str:
    text = str(text)
    text = HTML_RE.sub(" ", text)                 # remove HTML tags
    text = text.lower()                           # lowercase
    text = CONTRACTION_RE.sub(" not", text)       # "didn't" -> "did not"
    text = NON_ALPHA_RE.sub(" ", text)            # drop punctuation, digits
    tokens = text.split()                         # tokenize
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [lemmatize(t) for t in tokens]
    return " ".join(tokens)

demo = "<p>It STOPPED working after two days!! Not worth the money.</p>"
print("Before:", demo)
print("After :", clean_text(demo))

In [ ]:
df["clean_text"] = df["review_text"].apply(clean_text)
df[["review_text", "clean_text", "sentiment"]].head(10)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.25, random_state=RANDOM_STATE, stratify=df["label"]
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF matrix (train):", X_train_tfidf.shape)
print("Vocabulary size      :", len(tfidf.vocabulary_))

# Highest-weighted terms overall
weights = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top = np.argsort(weights)[::-1][:15]
print("\nTop terms:", [tfidf.get_feature_names_out()[i] for i in top])

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=1.0),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
}

results, preds = {}, {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    preds[name] = y_pred
    results[name] = {
        "Train accuracy": accuracy_score(y_train, model.predict(X_train_tfidf)),
        "Test accuracy": accuracy_score(y_test, y_pred),
        "Test F1 (macro)": f1_score(y_test, y_pred, average="macro"),
    }

pd.DataFrame(results).T.round(3)

In [ ]:
for name, y_pred in preds.items():
    print("=" * 60); print(name); print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=["negative", "positive"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
min_class = df["label"].value_counts().min()
n_splits = max(2, min(5, min_class))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

def make_pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
                     ("clf", clf)])

cv_rows = {}
for name, clf in [("Multinomial Naive Bayes", MultinomialNB()),
                  ("Logistic Regression", LogisticRegression(max_iter=1000))]:
    scores = cross_val_score(make_pipe(clf), df["clean_text"], df["label"], cv=cv, scoring="accuracy")
    cv_rows[name] = {"CV mean accuracy": scores.mean(), "CV std": scores.std(), "folds": n_splits}

pd.DataFrame(cv_rows).T.round(3)

In [ ]:
lr = models["Logistic Regression"]
names = np.array(tfidf.get_feature_names_out())
order = np.argsort(lr.coef_[0])

print("Most NEGATIVE terms:", list(names[order[:10]]))
print("Most POSITIVE terms:", list(names[order[-10:]][::-1]))

In [ ]:
# Refit final models on the full dataset
final_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_all = final_tfidf.fit_transform(df["clean_text"])
final_models = {
    "nb": MultinomialNB().fit(X_all, df["label"]),
    "lr": LogisticRegression(max_iter=1000).fit(X_all, df["label"]),
}
LABELS = {0: "negative", 1: "positive"}

def predict_sentiment(text: str, model: str = "lr", verbose: bool = True) -> dict:
    """Predict sentiment of a raw review string.

    model: 'nb' (Multinomial Naive Bayes) or 'lr' (Logistic Regression)
    """
    cleaned = clean_text(text)
    if not cleaned.strip():
        out = {"text": text, "label": "unknown", "confidence": None}
        if verbose: print("No usable words after cleaning:", repr(text))
        return out
    vec = final_tfidf.transform([cleaned])
    clf = final_models[model]
    proba = clf.predict_proba(vec)[0]
    pred = int(np.argmax(proba))
    out = {"text": text, "label": LABELS[pred], "confidence": float(proba[pred])}
    if verbose:
        icon = "😊" if pred == 1 else "😞"
        print(f"{icon} {out['label'].upper():8s} ({out['confidence']:.0%})  [{model}]  ->  {text}")
    return out

tests = [
    "Absolutely fantastic product, works great and arrived quickly!",
    "Terrible quality, it broke after one day. Total waste of money.",
    "Not worth the price, very disappointed.",
    "Easy to use and highly recommended.",
    "The package was damaged and support never replied.",
]
for t in tests:
    predict_sentiment(t, "lr")
    predict_sentiment(t, "nb")
    print()

In [ ]:
def interactive_test():
    print("Sentiment tester. Type a review ('quit' to exit).")
    while True:
        try:
            text = input("\nYour review > ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if text.lower() in {"quit", "exit", "q", ""}:
            print("Bye!"); break
        predict_sentiment(text, "lr")
        predict_sentiment(text, "nb")

# interactive_test()   # <- uncomment to run

In [ ]:
# Optional: button-style widget (works in Jupyter / Colab with ipywidgets)

    import ipywidgets as widgets
    from IPython.display import display, clear_output

    box = widgets.Text(placeholder="Type a review...", description="Review:", layout=widgets.Layout(width="80%"))
    btn = widgets.Button(description="Analyze", button_style="primary")
    out = widgets.Output()

    def _on_click(_):
        with out:
            clear_output()
            predict_sentiment(box.value, "lr")
            predict_sentiment(box.value, "nb")
    btn.on_click(_on_click)
    display(box, btn, out)
except ImportError:
    print("ipywidgets not installed; use interactive_test() instead.")

In [ ]:

    nltk.download("movie_reviews", quiet=True)
    from nltk.corpus import movie_reviews

    docs = [(" ".join(movie_reviews.words(fid)), cat) for cat in movie_reviews.categories()
            for fid in movie_reviews.fileids(cat)]
    big = pd.DataFrame(docs, columns=["review_text", "sentiment"])
    big["label"] = (big["sentiment"] == "pos").astype(int)
    big["clean_text"] = big["review_text"].apply(clean_text)

    Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(big["clean_text"], big["label"], test_size=0.2,
                                                  random_state=RANDOM_STATE, stratify=big["label"])
    vec_b = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
    Xb_tr_t, Xb_te_t = vec_b.fit_transform(Xb_tr), vec_b.transform(Xb_te)

    rows = {}
    for name, clf in [("Multinomial Naive Bayes", MultinomialNB()),
                      ("Logistic Regression", LogisticRegression(max_iter=1000))]:
        clf.fit(Xb_tr_t, yb_tr)
        rows[name] = {"Test accuracy": accuracy_score(yb_te, clf.predict(Xb_te_t)),
                      "Test F1 (macro)": f1_score(yb_te, clf.predict(Xb_te_t), average="macro")}
    display(pd.DataFrame(rows).T.round(3))
except Exception as e:
    print("Skipped larger benchmark:", e)

## 11. Summary

| Step | Choice |
|---|---|
| Cleaning | strip HTML, lowercase, remove punctuation/digits |
| Tokens | stopwords removed (negations kept), WordNet lemmatization |
| Features | `TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)` |
| Models | Multinomial Naive Bayes, Logistic Regression |
| Evaluation | hold-out accuracy/F1, confusion matrix, stratified k-fold CV |

**Findings (fill in after running your own numbers):**
- Compare Naive Bayes vs Logistic Regression test accuracy and CV mean ± std.
- Naive Bayes is fast and does well on small data; Logistic Regression usually wins as data grows and exposes interpretable word weights.
- The provided dataset is tiny, so treat its accuracy as indicative only; the larger-corpus benchmark gives a more reliable comparison.

**Possible improvements:** more labelled data, handling the `neutral` class (3-way classification), SpaCy lemmatization, hyperparameter search (`alpha`, `C`) with `GridSearchCV`.